# Split a PDF into documents

A tax return filed as one 40-page PDF holds 19 documents: the federal return, its schedules, supporting statements, and two state returns. Find where each one starts in two steps. First ask TypeLLM of every page, in parallel, whether it starts a document. Then go through the likely starts in order, and show each one beside the first page of the current document: does it start a new one, or continue it?

In [1]:
%pip install -q "typellm>=0.7.2" pymupdf huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [3]:
import json

import pymupdf
from huggingface_hub import hf_hub_download
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY, timeout=90)

# The return, its ground truth and the splitting rules, from the PoliTax-Split dataset.
REPO = "Extend-AI/PoliTax-Split"
NAME = "harris_2023_federal_state_returns.pdf"
pdf = hf_hub_download(REPO, f"pdfs/harris/{NAME}", repo_type="dataset")
truth = next(row for row in map(json.loads, open(hf_hub_download(REPO, "annotations.jsonl", repo_type="dataset")))
             if row["file_name"] == NAME)
taxonomy = json.load(open(hf_hub_download(REPO, "taxonomy.json", repo_type="dataset")))

doc = pymupdf.open(pdf)
print(f"{doc.page_count} pages, {len(truth['sub_documents'])} documents")

40 pages, 19 documents


## Write the rules

What counts as one document depends on you: is a schedule its own document, or part of the return? Are ten numbered statements one document or ten? Write it down. Here the rules come with the dataset: how to split, and a description of each kind of document.

In [4]:
RULES = ("A PDF holds several documents, one after another: tax forms, schedules and supporting pages. "
         "It is split into sub-documents by these rules:\n\n" + taxonomy["splitting_instructions"]
         + "\n\nThe sub-document types:\n\n" + "\n\n".join(f"{t['type']}: {t['description']}" for t in taxonomy["types"]))
print(f"{len(RULES):,} characters, {len(taxonomy['types'])} types")

19,948 characters, 51 types


## Ask each page

Send each page's image and its text with the rules, and ask one yes/no question with its probability. Every page that scores at least 0.2 is a likely start. The bar is low on purpose: a start missed here is missed for good, while an extra one is checked in the next step.

In [5]:
from concurrent.futures import ThreadPoolExecutor

INPUT, THINKING = [], []  # tokens of every call


def page_text(i):
    text = doc[i].get_text().strip()
    return text if len(text) >= 20 else "(no text layer; read the image)"


def image(i):
    return doc[i].get_pixmap(dpi=120).tobytes("png")


def yes(context, images, question, thinking=None):
    spec = {"type": "boolean", "return_probabilities": True, "instructions": question}
    if thinking:
        spec["thinking"] = thinking
    response = client.generate(context=context, images=images, questions={"yes": spec})
    INPUT.append(response.usage.input_tokens)
    THINKING.append(response.usage.thinking_tokens or 0)
    return response.result["yes"]["probabilities"][True]


def starts(i):
    context = RULES + "\n\nBelow is one page of the PDF: its image is attached, and its text follows.\n\n" + page_text(i)
    return yes(context, [image(i)], "Is this page the first page of a sub-document?")


with ThreadPoolExecutor(4) as pool:
    probs = list(pool.map(starts, range(doc.page_count)))
probs[0] = 1.0  # the first page always starts one
candidates = [i + 1 for i, p in enumerate(probs) if p >= 0.2]
print(f"{len(candidates)} likely starts: {candidates}")

34 likely starts: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 26, 27, 28, 29, 30, 31, 34, 36, 37, 38, 39, 40]


## Go through them in order

A page alone can't tell whether it starts a new document: a page titled "Statement 2" may continue the statements, and a page with a new form's title may be the next copy of the same form. Show it beside the first page of the current document, and let TypeLLM think as much as it needs. A new one becomes the current document; one that continues it is dropped, and the next likely start is asked against the same first page.

In [6]:
def new_after(first, later):
    context = (RULES + "\n\nBelow are two pages of the PDF, in order: the first page of a sub-document, then a later page. "
               "Their images are attached, and their text follows.\n\nThe first page's text:\n" + page_text(first - 1)
               + "\n\nThe later page's text:\n" + page_text(later - 1))
    return yes(context, [image(first - 1), image(later - 1)],
               "Does the later page start a new sub-document, rather than belong to the sub-document the first page starts?",
               thinking="auto")


found = [1]
for page in candidates[1:]:
    p = new_after(found[-1], page)
    print(f"page {found[-1]} vs {page}: {p:.2f} {'new' if p >= 0.5 else 'same'}")
    if p >= 0.5:
        found.append(page)

page 1 vs 2: 0.00 same
page 1 vs 3: 1.00 new
page 3 vs 4: 0.00 same
page 3 vs 5: 1.00 new
page 5 vs 6: 0.00 same
page 5 vs 7: 0.99 new
page 7 vs 8: 1.00 new
page 8 vs 9: 1.00 new
page 9 vs 10: 1.00 new
page 10 vs 11: 1.00 new
page 11 vs 13: 0.99 new
page 13 vs 14: 1.00 new
page 14 vs 15: 1.00 new
page 15 vs 16: 0.39 same
page 15 vs 17: 0.09 same
page 15 vs 18: 0.29 same
page 15 vs 19: 0.14 same
page 15 vs 20: 0.34 same
page 15 vs 21: 1.00 new
page 21 vs 22: 0.00 same
page 21 vs 24: 0.00 same
page 21 vs 26: 0.00 same
page 21 vs 27: 1.00 new
page 27 vs 28: 1.00 new
page 28 vs 29: 0.99 new
page 29 vs 30: 1.00 new
page 30 vs 31: 0.00 same
page 30 vs 34: 0.00 same
page 30 vs 36: 1.00 new
page 36 vs 37: 1.00 new
page 37 vs 38: 0.00 same
page 37 vs 39: 0.00 same
page 37 vs 40: 0.99 new


## Final result

In [7]:
ranges = list(zip(found, [s - 1 for s in found[1:]] + [doc.page_count]))
true_ranges = [(d["start_page"], d["end_page"]) for d in truth["sub_documents"]]
for d in truth["sub_documents"]:
    span = (d["start_page"], d["end_page"])
    name = d["identifier"] if d["type"] == "misc_form" else d["type"]  # misc_form: any other form, named by its identifier
    print(f"{name:25s} pages {span[0]}-{span[1]}: {'found' if span in ranges else 'missed'}")
print(f"\nfound but not there: {[r for r in ranges if r not in true_ranges]}")

cost = (sum(INPUT) * 0.05 + sum(THINKING) * 0.50) / 1e6
print(f"\n{len(INPUT)} calls, {sum(INPUT):,} input tokens, {sum(THINKING):,} thinking tokens: "
      f"${cost:.4f}, ${cost / doc.page_count:.5f} a page")

Form 1040                 pages 1-2: found
Schedule 1 (Form 1040)    pages 3-4: found
Schedule 2 (Form 1040)    pages 5-6: found
Schedule A (Form 1040)    pages 7-7: found
Schedule B (Form 1040)    pages 8-8: found
Schedule C (Form 1040)    pages 9-9: found
Schedule SE (Form 1040)   pages 10-10: found
Schedule H (Form 1040)    pages 11-12: found
Form 8959                 pages 13-13: found
Form 8960                 pages 14-14: found
supplement                pages 15-20: found
Form 540                  pages 21-26: found
CA Schedule W-2           pages 27-27: found
CA Schedule W-2           pages 28-28: found
CA Schedule W-2           pages 29-29: found
Schedule CA (540)         pages 30-35: found
Schedule S                pages 36-36: found
Form D-40                 pages 37-39: found
supplement                pages 40-40: found

found but not there: []

73 calls, 499,383 input tokens, 28,444 thinking tokens: $0.0392, $0.00098 a page
